<br>

We use [sklvq (Scikit-learning vector quantization)](https://sklvq.readthedocs.io/en/stable/), a scikit-learn compatible and expandable implementation of Learning Vector Quantization (LVQ) algorithms.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.cluster import KMeans
from sklvq.models import GLVQ
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 150
plt.style.use("ggplot")


df = pd.read_excel("./result/bayesian-clustered-bankClients.xlsx")
df.head()

In [ ]:
X = df.drop(columns=["persona_cluster"])
y = df["persona_cluster"]

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train LVQ model - con 1 prototype for each class (they can be more)
lvq = GLVQ(
    prototype_init="class-conditional-mean",
    prototype_n_per_class=1,  # 1 prototype per class
    random_state=42,
    solver_type="steepest-gradient-descent",
    solver_params={"max_runs": 100},
)

# Train the model
lvq.fit(X_train, y_train)

In [ ]:
# Make predictions
y_pred = lvq.predict(X_test)
y_pred_train = lvq.predict(X_train)

# Evaluate the model
print("\nModel Evaluation:")
print(f"Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Testing accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Get prototypes and their labels from the trained model
prototypes = lvq.prototypes_
prototype_labels = getattr(lvq, "prototypes_labels_", np.sort(np.unique(y_train)))

# Use real training feature names when available
if hasattr(X_train, "columns"):
    feature_names = list(X_train.columns)
else:
    feature_names = [f"feature_{i}" for i in range(prototypes.shape[1])]

# Safety check in case matrix width and feature list do not match
if len(feature_names) != prototypes.shape[1]:
    feature_names = [f"feature_{i}" for i in range(prototypes.shape[1])]

# Build prototype table directly in your dataset feature space
prototype_df = pd.DataFrame(prototypes, columns=feature_names)
prototype_df.insert(0, "Prototype_Label", prototype_labels)

# ## Visualizing the LVQ Prototypes

#

Now that we have trained our GLVQ model and extracted the prototypes, we need to understand what they represent.

Since our feature space is high-dimensional (combining continuous variables and one-hot encoded categories), we use two techniques to interpret the results:

1. **Feature Heatmap:** To analyze the exact "signature" (feature weights) of each persona's prototype.
2. **PCA 2D Projection:** To visualize the geometric position of the prototypes relative to the actual customer data points.


In [ ]:
import seaborn as sns

# ---------------------------------------------------------
# 1. Feature Heatmap of Prototypes
# ---------------------------------------------------------
plt.figure(figsize=(14, 10))

# Set 'Prototype_Label' as index and transpose so features are on the Y-axis and Personas on the X-axis
heatmap_data = prototype_df.set_index("Prototype_Label").T

# Plot the heatmap
sns.heatmap(
    heatmap_data,
    cmap="coolwarm",
    center=0,
    annot=True,  # Show numerical values
    fmt=".2f",  # Format to 2 decimal places
    cbar_kws={"label": "Feature Value / Centroid Weight"},
)

plt.title("LVQ Prototypes: Feature Signatures for each Persona", fontsize=16, pad=20)
plt.xlabel("Persona (Prototype Label)", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# ---------------------------------------------------------
# 2. PCA 2D Projection (Data vs. Prototypes)
# ---------------------------------------------------------
# Fit PCA on the training data to reduce to 2 dimensions for visualization
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train)

# Transform the learned prototypes using the EXACT SAME PCA mapping
prototypes_pca = pca.transform(prototypes)

plt.figure(figsize=(12, 8))

# Scatter plot of the actual training data points
sns.scatterplot(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    hue=y_train,
    palette="tab10",
    alpha=0.3,
    legend="full",
    edgecolor=None,
)

# Overlay the prototypes as large 'X' markers
plt.scatter(
    prototypes_pca[:, 0],
    prototypes_pca[:, 1],
    c="black",
    marker="X",
    s=250,
    linewidths=2,
    edgecolors="white",
    label="LVQ Prototypes",
    zorder=5,  # Ensure prototypes are drawn on top of the data points
)

# Formatting the plot
plt.title("LVQ Prototypes and Client Data in 2D PCA Space", fontsize=16)
plt.xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Persona Class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()